# 21 · Professional practice: functions, tests, Git and automated reports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/21_professional_practice.ipynb)

Notebooks are great for exploring, but code you rely on every month must be **reusable, tested and versioned**.
This notebook turns analysis code into a small tested module, shows the Git workflow every quant team uses, and
automates a monthly report, the kind of task a junior quant is typically given in the first weeks.

In [1]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

Ready. Data folder: ../data/


## 1. From copy-pasted cells to functions
If you have pasted the same three lines into five notebooks, it belongs in a function: one name, one docstring, one place to fix bugs. Good functions take inputs as arguments (no hidden globals), validate them, and return a result instead of printing it.

In [2]:
rets = pd.read_csv(DATA + "asset_returns_daily.csv", index_col=0, parse_dates=True)
port = rets.mean(axis=1)                       # an equal-weight portfolio to test our functions on

def annualized_return(returns, periods=252):
    """Geometric annual return of periodic returns."""
    r = pd.Series(returns, dtype=float).dropna()
    return (1 + r).prod() ** (periods / len(r)) - 1

print(f"equal-weight annual return: {annualized_return(port):.2%}")

equal-weight annual return: 4.49%


### Exercise 1

Write `annualized_vol(returns, periods=252)` returning the sample standard deviation (ddof=1) times √periods. It must accept a NumPy array or a pandas Series.

In [3]:
# Your code here

In [4]:
check("21.1");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 21.1: not answered yet. Store your answer in `annualized_vol`.


### Exercise 2

Write `max_drawdown(returns)` returning the largest peak-to-trough loss of cumulative wealth as a negative number (0 if there is none). Wealth starts at 1, so a loss on the very first day counts.

<details><summary>Hint</summary>

`wealth = np.concatenate([[1.0], np.cumprod(1 + np.asarray(returns))])`, then `(wealth / np.maximum.accumulate(wealth) - 1).min()`.

</details>

In [5]:
# Your code here

In [6]:
check("21.2");   # run me after your code: ✅ correct, ❌ try again, ⬜ not answered yet

⬜ Exercise 21.2: not answered yet. Store your answer in `max_drawdown`.


## 2. A module: one file, imported everywhere
`%%writefile` saves a cell as a file. Any notebook or script can then `import` it, so a fix in one place fixes every analysis.

In [7]:
%%writefile quantkit.py
"""Small, tested helpers for performance analysis."""
import numpy as np
import pandas as pd


def _clean(returns):
    r = pd.Series(returns, dtype=float).dropna()
    if r.empty:
        raise ValueError("returns is empty")
    return r


def annualized_return(returns, periods=252):
    r = _clean(returns)
    return float((1 + r).prod() ** (periods / len(r)) - 1)


def annualized_vol(returns, periods=252):
    return float(_clean(returns).std(ddof=1) * np.sqrt(periods))


def sharpe_ratio(returns, rf=0.0, periods=252):
    r = _clean(returns) - rf / periods
    return float(r.mean() / r.std(ddof=1) * np.sqrt(periods))


def max_drawdown(returns):
    wealth = np.concatenate([[1.0], (1 + _clean(returns)).cumprod().to_numpy()])
    return float((wealth / np.maximum.accumulate(wealth) - 1).min())


def summary(returns, rf=0.0, periods=252):
    return pd.Series({"ann. return": annualized_return(returns, periods), "ann. vol": annualized_vol(returns, periods),
                      "Sharpe": sharpe_ratio(returns, rf, periods), "max drawdown": max_drawdown(returns)})

Overwriting quantkit.py


In [8]:
import importlib, quantkit
importlib.reload(quantkit)                      # pick up edits without restarting
quantkit.summary(port, rf=0.005).round(4)

ann. return     0.0449
ann. vol        0.0924
Sharpe          0.4678
max drawdown   -0.3240
dtype: float64

## 3. Tests: prove the code does what you think
A test calls your function on a case where you know the answer. Run all tests after every change; if one fails, you find the bug before it reaches a client report. `pytest` finds every function whose name starts with `test_`.

In [9]:
%%writefile test_quantkit.py
import numpy as np
import pytest
import quantkit as qk


def test_vol_definition():
    r = np.array([0.01, -0.02, 0.015, 0.0, -0.005])
    assert qk.annualized_vol(r) == pytest.approx(np.std(r, ddof=1) * np.sqrt(252))


def test_drawdown_known_path():
    assert qk.max_drawdown([0.10, -0.50, 0.20]) == pytest.approx(-0.5)
    assert qk.max_drawdown([0.05, 0.05]) == 0.0


def test_constant_growth_return():
    daily = 1.10 ** (1 / 252) - 1
    assert qk.annualized_return(np.full(504, daily)) == pytest.approx(0.10)


def test_empty_input_raises():
    with pytest.raises(ValueError):
        qk.annualized_vol([np.nan])

Overwriting test_quantkit.py


In [10]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "test_quantkit.py"], capture_output=True, text=True).stdout[-400:])

....                                                                     [100%]
4 passed in 0.31s



**Try breaking it:** change `ddof=1` to `ddof=0` in `quantkit.py`, rerun the two cells above, and watch a test fail
with a message showing expected vs actual. That is the point: the test catches the mistake, not your client.
Good test cases: known textbook answers, edge cases (empty input, one observation, all losses), and invariants
(weights sum to 1, drawdown ≤ 0, no look-ahead).

## 4. Git: version control
Git records every change to your code with a message, so you can see what changed, why, and go back. GitHub hosts the repository so others (and recruiters) can see it. The core loop is just four commands.

In [11]:
import os, tempfile
repo = tempfile.mkdtemp()
def git(*args):
    out = subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()

git("init", "-q"); git("config", "user.email", "you@example.com"); git("config", "user.name", "Your Name")
with open(os.path.join(repo, "analysis.py"), "w") as f:
    f.write("RF = 0.005\n")
print(git("add", "analysis.py")); print(git("commit", "-q", "-m", "Add risk-free rate setting"))
with open(os.path.join(repo, "analysis.py"), "w") as f:
    f.write("RF = 0.01   # updated to current SARON level\n")
print("--- what changed (git diff):"); print(git("diff"))
git("commit", "-q", "-am", "Update risk-free rate"); print("--- history (git log):"); print(git("log", "--oneline"))



--- what changed (git diff):
diff --git a/analysis.py b/analysis.py
index d878e7a..6e1669a 100644
--- a/analysis.py
+++ b/analysis.py
@@ -1 +1 @@
-RF = 0.005
+RF = 0.01   # updated to current SARON level
--- history (git log):
dcbbe3a Update risk-free rate
bc21d3a Add risk-free rate setting


**The everyday workflow** (in a terminal or VS Code):
```bash
git clone https://github.com/<you>/<project>.git   # once
git pull                                          # get the latest version
git add src/ tests/ README.md                     # choose what to record
git commit -m "Add inverse-volatility weighting"  # record it with a clear message
git push                                          # upload to GitHub
```
**From Colab:** *File → Save a copy in GitHub* commits the open notebook to any repository you own (Colab asks to
connect your GitHub account once). Never commit passwords or API keys; keep them in Colab secrets or a `.env` file
listed in `.gitignore`.

**Good commit messages** say what and why in the imperative: “Fix look-ahead in monthly rebalance”, not “changes”.

## 5. Reproducibility checklist
Someone (including you in six months) must be able to rerun your analysis and get the same numbers.

In [12]:
import platform, sklearn, scipy
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)
CONFIG = {"start": "2015-01-01", "end": "2024-12-31", "rf": 0.005, "seed": 42}   # every setting in one place, at the top
rng = np.random.default_rng(CONFIG["seed"])                                       # fixed seed = same random draws every run
print("first random draw (identical on every run):", round(rng.normal(), 6))

Python 3.11.15 | pandas 3.0.6 | numpy 2.4.6 | scipy 1.17.1 | scikit-learn 1.9.1
first random draw (identical on every run): 0.304717


- Fix random seeds; put every setting in one config at the top.
- Record package versions (`pip freeze > requirements.txt`).
- Keep raw data read-only; every cleaning step in code, never by hand in Excel.
- Store data snapshots or the exact download date, because vendors revise history.

## 6. Automating the monthly report
A function that takes returns and produces a finished HTML report can run every month with one call, or on a schedule (cron, GitHub Actions, a scheduled Colab notebook).

In [13]:
import base64, io, datetime as dt
from IPython.display import HTML, display

def monthly_report(returns: pd.DataFrame, weights: pd.Series, path="monthly_report.html", rf=0.005):
    port = returns[weights.index] @ weights
    last = port.index.max(); month = port.loc[str(last.to_period("M"))]
    ytd = port.loc[str(last.year)]
    stats_ = pd.DataFrame({"Year to date": quantkit.summary(ytd, rf), "Since inception": quantkit.summary(port, rf)}).T
    stats_.insert(0, "total return", [(1 + ytd).prod() - 1, (1 + port).prod() - 1])
    stats_.loc["This month"] = [(1 + month).prod() - 1] + [np.nan] * 4      # annualising one month would mislead
    stats_ = stats_.loc[["This month", "Year to date", "Since inception"]]
    contrib = (returns.loc[month.index, weights.index] * weights).sum()
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    (1 + port.loc[str(last.year)]).cumprod().plot(ax=ax[0], title="Growth of 1, year to date")
    contrib.sort_values().plot.barh(ax=ax[1], title="Contribution to this month's return"); plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110); plt.close(fig)
    img = base64.b64encode(buf.getvalue()).decode()
    html = f"""<html><head><meta charset="utf-8"><title>Monthly report {last:%B %Y}</title>
    <style>body{{font-family:Arial,sans-serif;margin:32px;color:#222}} table{{border-collapse:collapse}}
    td,th{{border:1px solid #ccc;padding:4px 10px;text-align:right}}</style></head><body>
    <h2>Portfolio report: {last:%B %Y}</h2><p>Generated {dt.date.today()} from {len(port)} daily returns.</p>
    {stats_.style.format("{{:.2%}}", na_rep="–").format("{{:.2f}}", subset=["Sharpe"], na_rep="–").to_html()}
    <img src="data:image/png;base64,{img}" style="max-width:100%"></body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path, stats_

W = pd.Series({"CH_EQUITY": 0.20, "WORLD_EQUITY": 0.20, "EM_EQUITY": 0.05, "CHF_BONDS": 0.25,
               "GLOBAL_BONDS": 0.10, "GOLD": 0.05, "SWISS_REAL_ESTATE": 0.10, "COMMODITIES": 0.05})
path, tbl = monthly_report(rets, W)
print("written:", path); display(tbl.round(4))

written: monthly_report.html


,total return,ann. return,ann. vol,Sharpe,max drawdown
This month,-0.0417,NaN,NaN,NaN,NaN
Year to date,-0.0178,-0.0171,0.0859,-0.2160,-0.1358
Since inception,1.1353,0.0501,0.0865,0.5506,-0.3054


In Colab, open the file browser (folder icon, left) and download `monthly_report.html`, or send it by email from a
scheduled job. **Your turn (no checker):** add a 1-day 99% VaR line and the top-3 risk contributors to the report,
using functions from notebook 18, and add a test for each new function.

## 7. A code review checklist (use it on your own work)
- Does every function do one thing, with a docstring and no hidden globals?
- Is there a test for each function, including edge cases?
- Could any step use information from the future (look-ahead)?
- Are settings in one config and random seeds fixed?
- Would a colleague understand the code without you explaining it?

---
## Solutions

In [14]:
def annualized_vol(returns, periods=252):
    return float(np.std(np.asarray(returns, dtype=float), ddof=1) * np.sqrt(periods))

def max_drawdown(returns):
    wealth = np.concatenate([[1.0], np.cumprod(1 + np.asarray(returns, dtype=float))])
    return float((wealth / np.maximum.accumulate(wealth) - 1).min())

print(f"vol {annualized_vol(port):.2%} | max drawdown {max_drawdown(port):.2%}")
check_all("21")

vol 9.24% | max drawdown -32.40%
✅ Exercise 21.1: correct.
✅ Exercise 21.2: correct.

2 of 2 exercises correct.
